In [ ]:
#import
import sys
sys.path.append("..")
from topofinger import config
import numpy as np
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm
import os

import pickle as pk
from nilearn import plotting
import pandas as pd

import sys

from topofinger.hcp_IO import load_ts_schaefer300_dict, load_scaffold_schaefer300, load_ts_aal89_dict, load_scaffold
from topofinger.utils import threshold_matrix
from topofinger.utils_neuromaps_brain_multibrain import normal_view as normal_view_multibrain



%load_ext autoreload
%autoreload 2

#from visualize import threshold_adj_matrix

#paths
sup_folder = config.DATA_ROOT + "/"
ts_folder = sup_folder + "HCP/100unrelated/schaefer300/no_hp/ts/RL/"
fc_folder = sup_folder + "HCP/100unrelated/schaefer300/no_hp/fc/whole/"
sc_folder = sup_folder + "HCP/100unrelated/schaefer300/no_hp/scaffolds/whole/freq/"
figure_folder = config.FIGURES_RAW + "/final/fig1/"

# AAL89 paths
aal89_data_dir = sup_folder + "HCP/100unrelated/aal89/"
aal89_ts_folder = aal89_data_dir + "ts/TimeSeriesAAL/"
aal89_fc_folder = aal89_data_dir + "fc/whole/"
aal89_sc_folder = aal89_data_dir + "scaffolds/whole/freq/"

In [ ]:
#es_sig = np.random.random((300, ))
es_sig = np.random.random(300)

# Create the output directory if it doesn't exist
import os
os.makedirs(os.path.dirname(os.path.abspath(figure_folder + 'brain_regions.png')), exist_ok=True)

# Create a figure explicitly first
plt.figure(figsize=(10,8))
normal_view_multibrain(es_sig, edges=True, graymap_rev=True, exp_form=False, cmap='Blues',
                       alpha_graymap=0.99, center_cbar=True, parcellation=300)

# Ensure proper layout before saving
plt.tight_layout()

# Save the figure
plt.savefig(figure_folder + 'brain_regions.png', transparent=True, dpi=300)

# Show and close
plt.show()  # Display in notebook
plt.close()  # Properly close the figure

In [ ]:
## LOAD TS DATA ##

ts_data = load_ts_schaefer300_dict(ts_folder)
print('The dataset is composed of ' + str(len(ts_data.keys())) + ' subjects')
print('Each subject performed ' + str(len(ts_data['103414'].keys())) + ' trials')
print('Each trial contains ' + str(ts_data['103414'][0].shape[1]) +' time series (number of brain regions)')
print('The lenght of each time serie is ' + str(ts_data['103414'][0].shape[0]) + ' time points')


In [ ]:
# FIG1 panel A

subject = '100307'

fig = plt.figure(figsize=(8,2))
plt.plot(ts_data[subject][1][:,40], c='violet', alpha=0.7)
plt.plot(ts_data[subject][1][:,0], c='blue', alpha=0.7)
plt.plot(ts_data[subject][1][:,10], c='green', alpha=0.7)
plt.plot(ts_data[subject][1][:,30], c='firebrick', alpha=0.7)
#plt.plot(ts_data[subject][1][:,20], c='orange', alpha=0.7)
#plt.plot(ts_data['103414'][1][:,50], c='black', alpha=0.7)


plt.box(False)
plt.axis('off')
#plt.xlabel('Time', fontsize=12)
plt.tight_layout()
plt.savefig(figure_folder + 'fig1A.png', transparent=True, dpi=300)

plt.plot()
plt.show()

#check also schaefer300_selected/figure1

In [ ]:
subject = '101107'
plt.figure(figsize=(4,1))
plt.plot(ts_data[subject][1].T[9,:100], c='violet', alpha=0.7)
plt.axis('off')

plt.savefig('fig1_timeseries3.png', transparent=True, dpi=300)
plt.show()

In [ ]:
ts_data['100307'][0][:50,9].shape

In [ ]:
subject = '100307'

## LOAD FC DATA ##

with open(fc_folder + "whole_corrs.pck", "rb") as f:
    fc_data = pk.load(f)

print('The dictionary contains the funcional connectivity matrix for ' + str(len(fc_data.keys())) + ' subjects, undergoing ' + str(len(fc_data[subject].keys())) + ' trials each.')
print('Functional connectivity matrix dimension is ' + str(fc_data['103414'][0].shape[0]) + 'x' + str(fc_data['103414'][0].shape[1]))

np.fill_diagonal(fc_data[subject][1], 0)
plotting.plot_matrix(fc_data[subject][1], colorbar=False, cmap='Blues')
plt.savefig(figure_folder + 'fig1B.png', transparent=True, dpi=300)
plt.show()

In [ ]:
## LOAD AAL89 TS DATA ##

ts_aal89_data = load_ts_aal89_dict(aal89_ts_folder)
print('The AAL89 dataset is composed of ' + str(len(ts_aal89_data.keys())) + ' subjects')
if '103414' in ts_aal89_data and 0 in ts_aal89_data['103414']:
    print('Each subject performed ' + str(len(ts_aal89_data['103414'].keys())) + ' trials')
    print('Each trial contains ' + str(ts_aal89_data['103414'][0].shape[1] if ts_aal89_data['103414'][0].ndim > 1 else 89) +' time series (number of brain regions)')
    print('The length of each time series is ' + str(ts_aal89_data['103414'][0].shape[0]) + ' time points')
else:
    print('Data structure available in the AAL89 atlas')

In [ ]:
subject = '100307'

## LOAD FC DATA ##

with open(aal89_fc_folder + "whole_corrs.pck", "rb") as f:
    fc_data = pk.load(f)

print('The dictionary contains the funcional connectivity matrix for ' + str(len(fc_data.keys())) + ' subjects, undergoing ' + str(len(fc_data[subject].keys())) + ' trials each.')
print('Functional connectivity matrix dimension is ' + str(fc_data['103414'][0].shape[0]) + 'x' + str(fc_data['103414'][0].shape[1]))

np.fill_diagonal(fc_data[subject][1], 0)
thresholded = threshold_matrix(fc_data[subject][1], perc_to_keep=2)
plotting.plot_matrix(thresholded, colorbar=False, cmap='Blues', vmin=0.5)
plt.savefig(figure_folder + 'fig1C_FC.png', transparent=True, dpi=300)
plt.axis('off')
plt.show()

In [ ]:
#Plot aal scaffold in Oranges
subject = '100307'

aal89_scaffold = load_scaffold(aal89_sc_folder)

np.fill_diagonal(aal89_scaffold[subject][1], 0)
plotting.plot_matrix(aal89_scaffold[subject][1], colorbar=False, cmap='Oranges', vmin=0, vmax=3)
plt.savefig(figure_folder + 'fig1C_SC.png', transparent=True, dpi=300)
plt.axis('off')
plt.show()

## Panel C


In [ ]:
from sklearn.preprocessing import LabelEncoder

#open list of regions from txt
regions_list = []
with open(config.HCP_100 + '/schaefer300/Schaefer2018_300Parcels_7Networks_order.lut','r') as f:
    for line in f:
        elements = line.strip().split(' ')
        regions_list.append(elements[4])

#list of regions names        
regions7 = [reg.split('_')[2] for reg in regions_list]

label_encoder = LabelEncoder()

#list of regions encoded as numbers
regions7_numbers = label_encoder.fit_transform(regions7)
#dictionary with regions names and their corresponding numbers
dict_regions7 = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

#Save nodes csv
ids = range(300)
regions7
regions7_numbers

# Create a DataFrame with regions information
regions_df = pd.DataFrame({
    'Region_ID': ids,
    'Network_Name': regions7,
    'Network_Code': regions7_numbers
})

# Save the DataFrame to CSV
regions_df.to_csv(config.FIGURES_RAW + '/figure1/nodes.csv', index=False)


In [ ]:
sc_data = load_scaffold_schaefer300(sc_folder)
fc_matrix_th = threshold_matrix(fc_data['100307'][0],1)
sc_matrix = sc_data['100307'][0]

In [ ]:
# Generate list of tuples (source, target, weight) from the SC adjacency matrix
edges = []
for i in range(sc_matrix.shape[0]):
    for j in range(sc_matrix.shape[1]):
        if j>i and sc_matrix[i, j] > 0:  # Only include non-zero weights
            edges.append((i, j,'Undirected', sc_matrix[i, j]))

# Display the first few edges
print(f"Total number of edges: {len(edges)}")

# Save edges to csv if needed
edges_df = pd.DataFrame(edges, columns=['Source', 'Target', 'Type', 'Weight'])
edges_df.to_csv(config.FIGURES_RAW + '/figure1/edges_sc.csv', index=False)
print(edges_df.head(10))


# Generate list of tuples (source, target, weight) from the FC adjacency matrix
edges_fc = []
for i in range(fc_matrix_th.shape[0]):
    for j in range(fc_matrix_th.shape[1]):
        if j>i and fc_matrix_th[i, j] > 0:  # Only include non-zero weights
            edges_fc.append((i, j, "Undirected", fc_matrix_th[i, j]))
# Display the first few edges
print('')
print(f"Total number of edges: {len(edges_fc)}")
# Save edges to csv if needed
edges_fc_df = pd.DataFrame(edges_fc, columns=['Source', 'Target', 'Type', 'Weight'])
edges_fc_df.to_csv(config.FIGURES_RAW + '/figure1/edges_fc.csv', index=False)
print(edges_fc_df.head(10))




In [ ]:
edges_fc

In [ ]:
#Plot thresholded matrices

mat25 = threshold_matrix(fc_data[subject][1], 25, nan=True)
mat1 = threshold_matrix(fc_data[subject][1], 1, nan=True)
mat05 = threshold_matrix(fc_data[subject][1], 0.5, nan=True)

plotting.plot_matrix(mat25, colorbar=False, cmap='viridis')
plotting.plot_matrix(mat1, colorbar=False, cmap='viridis')
plotting.plot_matrix(mat05, colorbar=False, cmap='viridis')

In [ ]:
## LOAD SCAFFOLDS DATA ##

sc = load_scaffold_schaefer300(sc_folder)

sc_example = sc[subject][1]

sc_example = np.where(sc_example == 0, np.nan, sc_example)

plotting.plot_matrix(sc_example, colorbar=False, cmap='viridis')


In [ ]:
## LOAD SCAFFOLDS DATA ##

sc0 = np.where(sc['100408'][0] == 0, np.nan, sc['100408'][0])
sc1A = np.where(sc['100408'][1] == 0, np.nan, sc['100408'][1])
sc1B = np.where(sc['101309'][1] == 0, np.nan, sc['101309'][1])
sc1C = np.where(sc['101915'][1] == 0, np.nan, sc['101915'][1])
sc1D = np.where(sc['103111'][1] == 0, np.nan, sc['103111'][1])

plotting.plot_matrix(sc0, colorbar=False, cmap='viridis')
plotting.plot_matrix(sc1A, colorbar=False, cmap='viridis')
plotting.plot_matrix(sc1B, colorbar=False, cmap='viridis')
plotting.plot_matrix(sc1C, colorbar=False, cmap='viridis')
plotting.plot_matrix(sc1D, colorbar=False, cmap='viridis')
